In [ ]:
# Webスクレイピングに必要なライブラリ
import requests
from bs4 import BeautifulSoup
import time

In [2]:
# スクレイピング対象のURL
main_url = "https://www.musashino-u.ac.jp/"

In [3]:
# URLにアクセスしてHTMLを取得する関数
def get_all_links(url, visited=None):
    if visited is None:
        visited = set()  # 訪問済みURLの追跡用

    # URLにアクセスしてHTMLを取得
    response = requests.get(url)
    response.encoding = response.apparent_encoding
    soup = BeautifulSoup(response.text, "html.parser")

    # Key:URL Value:<title></title>の辞書を作成
    result = {}

    # <body>の中から<a>タグを取得
    url_list = soup.find('body').find_all('a', href=True)
    for link in url_list:
        href = link.get('href')
        # 内部リンクかつ未訪問のリンクかつPDF/XLSX/DOCXでないリンクのみ処理
        if (href.startswith('/') or href.startswith(main_url)) and not (href.endswith('.pdf') or href.endswith('.xlsx') or href.endswith('.docx') or href.endswith('.png') or href.endswith('.jpg') or href.endswith('doc')):
            # フルURLの作成
            full_url = href if href.startswith(main_url) else main_url + href.lstrip('/')

            # 重複チェック
            if full_url not in visited:
                visited.add(full_url)  # 訪問済みURLに追加
                try:
                    time.sleep(1)  # サーバー負荷軽減のための待機
                    res = requests.get(full_url)
                    res.encoding = res.apparent_encoding
                    res_soup = BeautifulSoup(res.text, "html.parser")
                    title = res_soup.find('title').text.strip() if res_soup.find('title') else "No title"
                    result[full_url] = title
                    print(f"{full_url} | Title: {title}")

                    # 再帰的にリンクを取得
                    result.update(get_all_links(full_url, visited))

                except Exception as e:
                    print(f"Failed to access {full_url}: {e}")

    return result

# すべてのリンクを取得
scraped_data = get_all_links(main_url)

# 出力
for key, value in scraped_data.items():
    print(f"{key} | {value}")

# 出力したURLの数を表示
print(f"\nTotal number of URLs retrieved: {len(scraped_data)}")

https://www.musashino-u.ac.jp/ | Title: 武蔵野大学
https://www.musashino-u.ac.jp/access.html | Title: 交通アクセス | 武蔵野大学
https://www.musashino-u.ac.jp/admission/request.html | Title: 資料請求 | 入試情報 | 武蔵野大学
https://www.musashino-u.ac.jp/contact.html | Title: お問い合わせ | 武蔵野大学
https://www.musashino-u.ac.jp/prospective-students.html | Title: 武蔵野大学で学びたい方 | 武蔵野大学
https://www.musashino-u.ac.jp/students.html | Title: 在学生の方 | 武蔵野大学
https://www.musashino-u.ac.jp/alumni.html | Title: 卒業生の方 | 武蔵野大学
https://www.musashino-u.ac.jp/parents.html | Title: 保護者の方 | 武蔵野大学
https://www.musashino-u.ac.jp/business.html | Title: 企業・研究者の方 | 武蔵野大学
https://www.musashino-u.ac.jp/guide/ | Title: 大学案内 | 武蔵野大学
https://www.musashino-u.ac.jp/guide/profile/ | Title: 大学紹介 | 大学案内 | 武蔵野大学
https://www.musashino-u.ac.jp/guide/activities/ | Title: 大学の取り組み | 大学案内 | 武蔵野大学
https://www.musashino-u.ac.jp/guide/campus/ | Title: キャンパス | 大学案内 | 武蔵野大学
https://www.musashino-u.ac.jp/guide/facility/ | Title: 附置機関・センター・附属施設 | 大学案内 | 武蔵野大学
https://www.mu